In [ ]:
from sklearn.linear_model import BayesianRidge
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

In [ ]:
forecasting_summary = pd.read_csv("latent_class_forecasting_summary.csv")

forecasting_summary["year"] = (forecasting_summary["year"].astype(str).str.split("/").str[0].astype(int))

forecasting_summary.head()

In [ ]:
INCLUDED_YEARS = [2016, 2017, 2018, 2019, 2020, 2021, 2022]

FORECAST_YEARS = [2023, 2024, 2025, 2026, 2027]

In [ ]:
def bayesian_ridge_forecast_classes(forecasting_df, target_col, included_years, forecast_years):

    output = []

    X_train = (np.array(included_years) - included_years[0]).reshape(-1, 1)
    X_forecast = (np.array(forecast_years) - included_years[0]).reshape(-1, 1)

    for cls in sorted(forecasting_df["Class"].unique()):

        tmp = (forecasting_df[(forecasting_df["Class"] == cls) & (forecasting_df["year"].isin(included_years))].sort_values("year"))

        if len(tmp) != len(included_years):
            print(f"Skipping Class {cls}: missing years.")
            continue

        y_train = tmp[target_col].values

        model = BayesianRidge(alpha_1=1e-6, alpha_2=1e-6, lambda_1=1e-6, lambda_2=1e-6, max_iter=300, tol=1e-3)

        model.fit(X_train, y_train)

        r2 = model.score(X_train, y_train)

        y_forecast, forecast_std = model.predict(X_forecast, return_std=True)

        years_full = included_years + forecast_years

        values_full = np.concatenate((y_train, y_forecast))

        error_full = np.concatenate((np.zeros(len(y_train)), forecast_std))

        for year, value, error in zip(years_full, values_full, error_full):
            output.append({"Class": cls, "year": year, "value": value, "error": error, "R2": r2})

    return pd.DataFrame(output)

In [ ]:
target_col = "Mean_MEMS7_IN"

forecast_df = bayesian_ridge_forecast_classes(forecasting_summary, target_col=target_col, included_years=INCLUDED_YEARS, forecast_years=FORECAST_YEARS)

forecast_df.to_csv(f"lca_{target_col.lower()}_bayesian_forecast.csv", index=False)

forecast_df.head()

In [ ]:
class_names = {
    0:  "Young highly educated professionals",
    1:  "Lone mothers in professional work",
    2:  "Young adults living with parents",
    3:  "Young students and early job seekers",
    4:  "Young affluent professional women",
    5:  "Young working professionals",
    6:  "Retired affluent older adults",
    7:  "Disabled mid-life adults",
    8:  "Young students living with parents",
    9:  "Established professional adults",
    10: "Mid-life professional women",
    11: "Retired lower-middle adults",
    12: "University students in shared housing",
    13: "Professional fathers",
    14: "Working family fathers",
    15: "Older established professionals",
    16: "Older mixed-status families",
    17: "Older women working part-time",
    18: "Professional mothers",
    19: "Lower socioeconomic family mothers",
    20: "Older working adults",
    21: "Affluent oldest retirees",
    22: "Disabled oldest retirees",
    23: "Highly educated professional fathers",
    24: "Economically disadvantaged unemployed adults"}

In [ ]:
plot_df = forecast_df.copy()

plot_df["ClassName"] = plot_df["Class"].map(class_names)

plot_df["Forecast"] = plot_df["year"].isin(FORECAST_YEARS)

colours = (["#808080"] * len(INCLUDED_YEARS) + ["#00BFFF"] * len(FORECAST_YEARS))

g = sns.relplot(kind="scatter",
                data=plot_df,
                x="year",
                y="value",
                col="ClassName",
                col_wrap=5,
                height=2.3,
                aspect=1.4,
                s=50,
                palette="RdYlGn",
                hue="value",
                legend=False,
                edgecolors=colours,
                linewidth=1,
                zorder=2)

for cls_name, ax in g.axes_dict.items():

    class_df = (plot_df[plot_df["ClassName"] == cls_name].sort_values("year"))

    sns.regplot(data=class_df,
                x="year",
                y="value",
                scatter=False,
                ci=None,
                color="#00BFFF",
                line_kws={"alpha":0.4, "zorder":0},
                ax=ax)

    future = class_df[class_df["year"].isin(FORECAST_YEARS)]

    ax.errorbar(x=future["year"],
                y=future["value"],
                yerr=1.96*future["error"],
                fmt="none",
                color="#00BFFF",
                alpha=0.2,
                capsize=3,
                zorder=1)

    ax.axvline(INCLUDED_YEARS[-1], linestyle=":", color="black", alpha=0.5)

g.set_titles("{col_name}", weight="bold", size=9)

g.set(xlabel="Year", ylabel=target_col.replace("_", " ").title())

g.set_xticklabels([])
g.set_yticklabels([])

plt.subplots_adjust(top=0.94)

g.figure.suptitle(f"Bayesian Ridge Forecasts of {target_col.replace("_", " ").title()} by Latent Class", fontsize=16, fontweight="bold")

plt.savefig(f"lca_{target_col.lower()}_bayesian_forecast.png", dpi=300, bbox_inches="tight")

plt.show()